# Policy Gradient Learning (REINFORCE)

This notebook walks through **policy gradient reinforcement learning**, starting from the theory and ending with a working REINFORCE agent that learns to balance the CartPole-v1 environment.

We will cover:
1. The policy gradient objective and its gradient estimator
2. REINFORCE (Monte-Carlo policy gradient) with return normalization
3. A variance-reduced variant with a learned value baseline
4. Training, evaluation, and visualization

Runs on Google Colab (CPU is fine for CartPole).

## 1. Background

A **policy** $\pi_\theta(a\mid s)$ is a distribution over actions parameterized by $\theta$. The RL objective is the expected return

$$J(\theta) = \mathbb{E}_{\tau\sim\pi_\theta}\Big[\sum_{t=0}^{T} \gamma^t r_t \Big].$$

The **policy gradient theorem** gives

$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau\sim\pi_\theta}\Big[\sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t\mid s_t)\; G_t \Big],$$

where $G_t = \sum_{k=t}^{T} \gamma^{k-t} r_k$ is the discounted return from step $t$.

**REINFORCE** is the Monte-Carlo estimator of this gradient. Variance is reduced by subtracting a baseline $b(s_t)$ (often a learned value $V_\phi(s_t)$), giving the advantage $A_t = G_t - b(s_t)$.

## 2. Setup

Install dependencies. `gymnasium` is the maintained fork of OpenAI Gym.

In [ ]:
!pip install -q gymnasium[classic-control] torch numpy matplotlib

In [ ]:
import random
from collections import deque

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print('Using device:', device)

## 3. Environment

CartPole-v1: 4-dim continuous state, 2 discrete actions (push left/right). The episode ends when the pole falls or after 500 steps; every step yields reward 1.

In [ ]:
env = gym.make('CartPole-v1')
obs_dim = env.observation_space.shape[0]
n_actions = env.action_space.n
print(f'obs_dim={obs_dim}, n_actions={n_actions}')

## 4. Policy network

A small MLP that maps state to action logits. `Categorical` gives us `sample()` and `log_prob()` in one object.

In [ ]:
class PolicyNet(nn.Module):
    def __init__(self, obs_dim, n_actions, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x):
        return self.net(x)

    def act(self, state):
        state = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        logits = self.forward(state)
        dist = Categorical(logits=logits)
        action = dist.sample()
        return action.item(), dist.log_prob(action).squeeze(0)

## 5. Returns

Compute discounted returns $G_t$ backwards through the episode.

In [ ]:
def discounted_returns(rewards, gamma):
    returns = []
    G = 0.0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.append(G)
    returns.reverse()
    return torch.tensor(returns, dtype=torch.float32, device=device)

## 6. REINFORCE training loop

For each episode: roll out the policy, compute returns, and minimize

$$L(\theta) = -\frac{1}{T}\sum_t \log \pi_\theta(a_t\mid s_t)\; \hat{A}_t,$$

where $\hat{A}_t$ is the normalized return (mean 0, std 1) as a simple variance-reduction trick.

In [ ]:
def train_reinforce(env, episodes=600, gamma=0.99, lr=1e-2, log_every=25):
    policy = PolicyNet(obs_dim, n_actions).to(device)
    optimizer = torch.optim.Adam(policy.parameters(), lr=lr)

    history = []
    recent = deque(maxlen=50)

    for ep in range(1, episodes + 1):
        state, _ = env.reset(seed=SEED + ep)
        log_probs, rewards = [], []
        done = False
        while not done:
            action, logp = policy.act(state)
            state, reward, terminated, truncated, _ = env.step(action)
            log_probs.append(logp)
            rewards.append(reward)
            done = terminated or truncated

        returns = discounted_returns(rewards, gamma)
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        log_probs = torch.stack(log_probs)
        loss = -(log_probs * returns).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total = sum(rewards)
        history.append(total)
        recent.append(total)
        if ep % log_every == 0:
            print(f'ep {ep:4d} | return {total:6.1f} | avg50 {np.mean(recent):6.1f} | loss {loss.item():+.3f}')

    return policy, history

In [ ]:
policy, history = train_reinforce(env)

## 7. Learning curve

In [ ]:
def plot_history(history, title='REINFORCE on CartPole-v1'):
    window = 25
    smoothed = np.convolve(history, np.ones(window) / window, mode='valid')
    plt.figure(figsize=(9, 4))
    plt.plot(history, alpha=0.3, label='episode return')
    plt.plot(range(window - 1, len(history)), smoothed, label=f'{window}-ep moving avg')
    plt.xlabel('episode')
    plt.ylabel('return')
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

plot_history(history)

## 8. REINFORCE with a learned value baseline

Using $\hat{A}_t = G_t - V_\phi(s_t)$ reduces variance without adding bias. We fit $V_\phi$ by regression against $G_t$.

In [ ]:
class ValueNet(nn.Module):
    def __init__(self, obs_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_reinforce_with_baseline(env, episodes=600, gamma=0.99, lr_pi=1e-2, lr_v=1e-2, log_every=25):
    policy = PolicyNet(obs_dim, n_actions).to(device)
    value = ValueNet(obs_dim).to(device)
    opt_pi = torch.optim.Adam(policy.parameters(), lr=lr_pi)
    opt_v = torch.optim.Adam(value.parameters(), lr=lr_v)

    history = []
    recent = deque(maxlen=50)

    for ep in range(1, episodes + 1):
        state, _ = env.reset(seed=SEED + ep)
        states, log_probs, rewards = [], [], []
        done = False
        while not done:
            action, logp = policy.act(state)
            states.append(state)
            log_probs.append(logp)
            state, reward, terminated, truncated, _ = env.step(action)
            rewards.append(reward)
            done = terminated or truncated

        returns = discounted_returns(rewards, gamma)
        states_t = torch.as_tensor(np.array(states), dtype=torch.float32, device=device)
        values = value(states_t)

        advantages = (returns - values).detach()
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        log_probs = torch.stack(log_probs)
        pi_loss = -(log_probs * advantages).mean()
        v_loss = F.mse_loss(values, returns)

        opt_pi.zero_grad(); pi_loss.backward(); opt_pi.step()
        opt_v.zero_grad(); v_loss.backward(); opt_v.step()

        total = sum(rewards)
        history.append(total)
        recent.append(total)
        if ep % log_every == 0:
            print(f'ep {ep:4d} | return {total:6.1f} | avg50 {np.mean(recent):6.1f} | pi {pi_loss.item():+.3f} | v {v_loss.item():.2f}')

    return policy, value, history

In [ ]:
policy_b, value_b, history_b = train_reinforce_with_baseline(env)
plot_history(history_b, title='REINFORCE + value baseline on CartPole-v1')

## 9. Evaluate the trained policy

Run greedy (argmax) and stochastic rollouts and report mean return.

In [ ]:
def evaluate(policy, env, episodes=20, greedy=True):
    returns = []
    for i in range(episodes):
        state, _ = env.reset(seed=10_000 + i)
        done = False
        total = 0.0
        while not done:
            state_t = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
            with torch.no_grad():
                logits = policy(state_t)
            if greedy:
                action = int(torch.argmax(logits, dim=-1).item())
            else:
                action = int(Categorical(logits=logits).sample().item())
            state, reward, terminated, truncated, _ = env.step(action)
            total += reward
            done = terminated or truncated
        returns.append(total)
    return float(np.mean(returns)), float(np.std(returns))

mean_g, std_g = evaluate(policy_b, env, greedy=True)
mean_s, std_s = evaluate(policy_b, env, greedy=False)
print(f'greedy    : {mean_g:.1f} ± {std_g:.1f}')
print(f'stochastic: {mean_s:.1f} ± {std_s:.1f}')

## 10. Where to go next

- **Actor-Critic / A2C** — bootstrap $V_\phi$ to form a TD advantage instead of using full Monte-Carlo returns.
- **GAE** (Generalized Advantage Estimation) — interpolates between TD and Monte-Carlo.
- **PPO** — clipped surrogate objective for stable large updates; today's default policy-gradient algorithm.
- **Continuous actions** — replace `Categorical` with `Normal` and output mean/log-std.
- **Harder envs** — try `LunarLander-v2`, `Acrobot-v1`, or MuJoCo tasks.